# Google Trends, HCUP, & U.S. Census Bureau Data Collection
**Author:** J. Casey Brookshier  
**Date:** July 15, 2026

## Objective

Collect, clean, and merge Google Trends, HCUP, and U.S. Census Bureau data to build a state-year dataset for predicting future inpatient psychiatric admissions.

### Data Sources
**Google Trends:** State-level suicide/crisis search interest (predictors)
  **HCUP:** Annual Mental Health/Substance Use inpatient admissions (outcome)
  **U.S. Census Bureau:** State population estimates (normalization)

### Workflow
**Collect → Clean → Merge → Model → Evaluate**

The resulting dataset supports regression, Random Forest, XGBoost, PCA, hierarchical clustering, and SHAP feature importance analyses.

In [ ]:
# imports & project paths


from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

PROJECT_ROOT = Path.cwd().resolve()

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data:", RAW_DIR)
print("Processed data:", PROCESSED_DIR)


In [ ]:
#define input output files

GT_RAW_FILE = RAW_DIR / "google_trends_raw_progress.csv"
HCUP_RAW_FILE = RAW_DIR / "DownloadTable_StatePayerIP_2026-03-20.xls"
CENSUS_2020_FILE = RAW_DIR / "nst-est2020.xlsx"
CENSUS_2025_FILE = RAW_DIR / "NST-EST2025-POP.xlsx"

OUTPUT_FILE = (
    PROCESSED_DIR
    / "team_rho_state_year_prediction_dataset.csv"
)

SEARCH_TERMS = [
    "suicidal thoughts",
    "suicide hotline",
    "self harm",
    "mental health crisis",
    "suicide prevention",
    "crisis hotline",
    "psychiatric hospital",
    "depression help",
]

YEARS = range(2013, 2024)

In [ ]:
# verify repo structure

required_dirs = [
    RAW_DIR,
    PROCESSED_DIR,
]

for directory in required_dirs:
    if not directory.exists():
        raise FileNotFoundError(f"Missing directory: {directory}")

print("Repository structure verified.")

In [ ]:
# load google trends data

if not GT_RAW_FILE.exists():
    raise FileNotFoundError(
        f"Missing Google Trends file: {GT_RAW_FILE}"
    )

google = pd.read_csv(GT_RAW_FILE)

google["State"] = google["State"].astype(str).str.strip()
google["Year"] = pd.to_numeric(
    google["Year"],
    errors="coerce"
)

google = google[
    google["Year"].between(2013, 2023)
].copy()

google["Year"] = google["Year"].astype(int)

for term in SEARCH_TERMS:
    if term not in google.columns:
        google[term] = np.nan

    google[term] = pd.to_numeric(
        google[term],
        errors="coerce"
    )

google_year = (
    google
    .groupby(
        ["State", "Year"],
        as_index=False
    )[SEARCH_TERMS]
    .mean()
)

google_year = (
    google_year
    .sort_values(["State", "Year"])
    .reset_index(drop=True)
)

print("Google Trends shape:", google_year.shape)
print("States:", google_year["State"].nunique())
print("Years:", sorted(google_year["Year"].unique()))

In [ ]:
#standarize google data

google["State"] = google["State"].astype(str).str.strip()
google["Year"] = pd.to_numeric(google["Year"], errors="coerce")

google = google[
    google["Year"].between(2013, 2023)
].copy()

google["Year"] = google["Year"].astype(int)

for term in SEARCH_TERMS:
    if term not in google.columns:
        google[term] = np.nan

    google[term] = pd.to_numeric(
        google[term],
        errors="coerce"
    )

if "Search_Term" in google.columns:
    google_year = (
        google
        .groupby(["State", "Year"], as_index=False)[SEARCH_TERMS]
        .mean()
    )
else:
    google_year = google[
        ["State", "Year"] + SEARCH_TERMS
    ].copy()

google_year = (
    google_year
    .sort_values(["State", "Year"])
    .drop_duplicates(["State", "Year"])
    .reset_index(drop=True)
)

print("Google Trends shape:", google_year.shape)
print("States:", google_year["State"].nunique())
print("Years:", sorted(google_year["Year"].unique()))

In [ ]:
# create google trend lag variables

for term in SEARCH_TERMS:
    google_year[f"{term}_lag1"] = (
        google_year
        .groupby("State")[term]
        .shift(1)
    )

google_year.head()

In [ ]:
# load HCUP admission data
if not HCUP_RAW_FILE.exists():
    raise FileNotFoundError(
        f"Missing HCUP file: {HCUP_RAW_FILE}"
    )

raw_hcup = pd.read_excel(
    HCUP_RAW_FILE,
    sheet_name="Data",
    header=None
)

header_row = None

for i in range(len(raw_hcup)):
    row = raw_hcup.iloc[i].astype(str).tolist()

    if (
        "State" in row
        and "Hospitalization Type" in row
        and "Expected Payer" in row
    ):
        header_row = i
        break

if header_row is None:
    raise ValueError("HCUP header row not found.")

hcup = pd.read_excel(
    HCUP_RAW_FILE,
    sheet_name="Data",
    header=header_row
)

hcup.columns = (
    hcup.columns
    .astype(str)
    .str.strip()
)

quarter_cols = [
    c for c in hcup.columns
    if (
        isinstance(c, str)
        and len(c) >= 7
        and c[:4].isdigit()
        and "Q" in c
    )
]

for c in quarter_cols:
    hcup[c] = (
        hcup[c]
        .astype(str)
        .str.replace(",", "", regex=False)
        .replace(
            ["nan", "None", ""],
            np.nan
        )
    )

    hcup[c] = pd.to_numeric(
        hcup[c],
        errors="coerce"
    )

analysis_quarters = [
    c for c in quarter_cols
    if any(
        str(year) in c
        for year in YEARS
    )
]

hcup = hcup[
    [
        "State",
        "Hospitalization Type",
        "Expected Payer",
        *analysis_quarters
    ]
].copy()

mh = hcup[
    hcup["Hospitalization Type"]
    == "Mental Health/Substance Use"
].copy()

summary_patterns = (
    "Combined|"
    "Sum|"
    "All expected"
)

mh = mh[
    ~mh["Expected Payer"]
    .str.contains(
        summary_patterns,
        case=False,
        regex=True,
        na=False
    )
]

valid_payers = [
    "Medicare, age 65+",
    "Medicaid, age 19-64",
    "Private, age 19-64",
    "Self-Pay/No Charge, age 19-64"
]

mh = mh[
    mh["Expected Payer"].isin(valid_payers)
].copy()

mh_state_quarter = (
    mh
    .groupby("State")[analysis_quarters]
    .sum(min_count=1)
    .reset_index()
)

annual_records = []

for year in YEARS:

    year_cols = [
        c for c in analysis_quarters
        if str(year) in c
    ]

    temp = mh_state_quarter[
        ["State", *year_cols]
    ].copy()

    temp["Year"] = year

    temp["mental_health_admissions"] = (
        temp[year_cols]
        .sum(
            axis=1,
            min_count=1
        )
    )

    annual_records.append(
        temp[
            [
                "State",
                "Year",
                "mental_health_admissions"
            ]
        ]
    )

hcup = pd.concat(
    annual_records,
    ignore_index=True
)

hcup["State"] = (
    hcup["State"]
    .astype(str)
    .str.strip()
)

hcup["Year"] = hcup["Year"].astype(int)

hcup = hcup[
    hcup["mental_health_admissions"].notna()
    & (
        hcup["mental_health_admissions"] >= 0
    )
].copy()

hcup = (
    hcup
    .drop_duplicates(["State", "Year"])
    .sort_values(["State", "Year"])
    .reset_index(drop=True)
)

print("HCUP shape:", hcup.shape)
print("States:", hcup["State"].nunique())
print("Years:", sorted(hcup["Year"].unique()))

In [ ]:
# validate HCUP admissions data

if hcup.duplicated(["State", "Year"]).any():
    raise ValueError("HCUP contains duplicate State-Year rows.")

if hcup["mental_health_admissions"].isna().any():
    raise ValueError("HCUP contains missing admission values.")

if (hcup["mental_health_admissions"] < 0).any():
    raise ValueError("HCUP contains negative admission values.")

print("HCUP validation passed.")

In [ ]:
# load census population data
if not CENSUS_2020_FILE.exists():
    raise FileNotFoundError(
        f"Missing Census file: {CENSUS_2020_FILE}"
    )

if not CENSUS_2025_FILE.exists():
    raise FileNotFoundError(
        f"Missing Census file: {CENSUS_2025_FILE}"
    )

old_raw = pd.read_excel(
    CENSUS_2020_FILE,
    header=None
)

new_raw = pd.read_excel(
    CENSUS_2025_FILE,
    header=None
)

old_start = old_raw[
    old_raw.iloc[:, 0]
    .astype(str)
    .str.startswith(".Alabama")
].index[0]

new_start = new_raw[
    new_raw.iloc[:, 0]
    .astype(str)
    .str.startswith(".Alabama")
].index[0]

old_years = [
    "State",
    "2010",
    "Base",
    "2010",
    "2011",
    "2012",
    "2013",
    "2014",
    "2015",
    "2016",
    "2017",
    "2018",
    "2019",
    "2020_Apr",
    "2020"
]

old = old_raw.iloc[old_start:].copy()
old.columns = old_years

old = old[
    [
        "State",
        "2013",
        "2014",
        "2015",
        "2016",
        "2017",
        "2018",
        "2019",
        "2020"
    ]
]

new_years = [
    "State",
    "Base",
    "2020",
    "2021",
    "2022",
    "2023",
    "2024",
    "2025"
]

new = new_raw.iloc[new_start:].copy()
new.columns = new_years

new = new[
    [
        "State",
        "2021",
        "2022",
        "2023"
    ]
]

wide = old.merge(
    new,
    on="State",
    how="left"
)

wide["State"] = (
    wide["State"]
    .astype(str)
    .str.replace(
        ".",
        "",
        regex=False
    )
    .str.strip()
)

exclude = {
    "United States",
    "Northeast",
    "Midwest",
    "South",
    "West",
    "Puerto Rico"
}

wide = wide[
    ~wide["State"].isin(exclude)
].copy()

census = wide.melt(
    id_vars="State",
    var_name="Year",
    value_name="state_population"
)

census["Year"] = pd.to_numeric(
    census["Year"],
    errors="coerce"
)

census["state_population"] = pd.to_numeric(
    census["state_population"],
    errors="coerce"
)

census = census[
    census["Year"].between(2013, 2023)
    & census["State"].notna()
    & census["state_population"].notna()
].copy()

census["Year"] = census["Year"].astype(int)

census = (
    census[
        [
            "State",
            "Year",
            "state_population"
        ]
    ]
    .drop_duplicates(["State", "Year"])
    .sort_values(["State", "Year"])
    .reset_index(drop=True)
)

print("Census shape:", census.shape)
print("States:", census["State"].nunique())
print("Years:", sorted(census["Year"].unique()))

In [ ]:
# validate census data

if census.duplicated(["State", "Year"]).any():
    raise ValueError("Census contains duplicate State-Year rows.")

if (census["state_population"] <= 0).any():
    raise ValueError("Census contains non-positive population values.")

print("Census validation passed.")

In [ ]:
# merge data sets
dataset = (
    google_year
    .merge(
        hcup,
        on=["State", "Year"],
        how="inner",
        validate="one_to_one",
    )
    .merge(
        census,
        on=["State", "Year"],
        how="inner",
        validate="one_to_one",
    )
)

print("Merged shape:", dataset.shape)
print("States:", dataset["State"].nunique())
print("Years:", sorted(dataset["Year"].unique()))

In [ ]:
# calculate admission rates

dataset["admission_rate_per_100k"] = (
    dataset["mental_health_admissions"]
    / dataset["state_population"]
    * 100_000
)

dataset["admission_rate_per_100k"].describe()

In [ ]:
# keep predictive modeling period

dataset = dataset[
    dataset["Year"].between(2014, 2023)
].copy()

dataset = dataset.sort_values(
    ["State", "Year"]
).reset_index(drop=True)

print("Final modeling period:", dataset["Year"].min(), "to", dataset["Year"].max())

In [ ]:
#select final variables

lag_columns = [
    f"{term}_lag1"
    for term in SEARCH_TERMS
]

final_columns = [
    "State",
    "Year",
    "state_population",
    "mental_health_admissions",
    "admission_rate_per_100k",
    *lag_columns,
]

dataset = dataset[final_columns].copy()

dataset.head()

In [ ]:
# validate final data set

expected_years = set(range(2014, 2024))

print("=" * 70)
print("FINAL DATASET VALIDATION")
print("=" * 70)

print("\nShape:")
print(dataset.shape)

print("\nStates:")
print(dataset["State"].nunique())

print("\nYears:")
print(sorted(dataset["Year"].unique()))

print("\nDuplicate State-Year rows:")
print(
    dataset.duplicated(
        ["State", "Year"]
    ).sum()
)

print("\nMissing values:")
print(dataset.isna().sum())

print("\nAdmission distribution:")
print(
    dataset["mental_health_admissions"].describe()
)

print("\nAdmission rate distribution:")
print(
    dataset["admission_rate_per_100k"].describe()
)

In [ ]:
#check state-year structure

state_year_check = (
    dataset
    .groupby("State")["Year"]
    .agg(["min", "max", "count"])
)

print(state_year_check)

if dataset.duplicated(["State", "Year"]).any():
    raise ValueError(
        "Final dataset contains duplicate State-Year observations."
    )

if dataset["state_population"].isna().any():
    raise ValueError(
        "Final dataset contains missing population values."
    )

if dataset["mental_health_admissions"].isna().any():
    raise ValueError(
        "Final dataset contains missing outcome values."
    )

if (dataset["state_population"] <= 0).any():
    raise ValueError(
        "Final dataset contains invalid population values."
    )

if (dataset["mental_health_admissions"] < 0).any():
    raise ValueError(
        "Final dataset contains negative admission values."
    )

print("\nFinal validation passed.")

In [ ]:
# check final data set

display(
    dataset
    .sort_values(["State", "Year"])
    .head(25)
)

In [ ]:
# save to processed data

dataset.to_csv(
    OUTPUT_FILE,
    index=False
)

print(f"Saved: {OUTPUT_FILE}")
print(f"Rows: {len(dataset):,}")
print(f"Columns: {len(dataset.columns)}")

In [ ]:
# confirm saved file available

check = pd.read_csv(OUTPUT_FILE)

print("Saved file:", OUTPUT_FILE)
print("Shape:", check.shape)
print("Duplicate State-Year rows:", check.duplicated(["State", "Year"]).sum())

display(check.head())

In [ ]:
# final summary

print("=" * 70)
print("TEAM RHO DATASET BUILD COMPLETE")
print("=" * 70)

print(f"Output: {OUTPUT_FILE.relative_to(PROJECT_ROOT)}")
print(f"Rows: {len(dataset):,}")
print(f"Columns: {len(dataset.columns)}")
print(f"States: {dataset['State'].nunique()}")
print(f"Years: {dataset['Year'].min()}-{dataset['Year'].max()}")
print(
    f"Duplicate State-Year rows: "
    f"{dataset.duplicated(['State', 'Year']).sum()}"
)
print(
    f"Missing outcome values: "
    f"{dataset['mental_health_admissions'].isna().sum()}"
)
print(
    f"Missing population values: "
    f"{dataset['state_population'].isna().sum()}"
)